# Step 13: Test the Deployed Endpoint

**SageMaker Unified Studio Component**: Inference Endpoints (Testing)

**What you'll learn**: Test your deployed model with various scenarios and understand the API

In [ ]:
import boto3
import json
import pandas as pd
from IPython.display import display

# Initialize SageMaker runtime client
runtime = boto3.client('sagemaker-runtime', region_name='eu-west-1')
endpoint_name = 'machine-overheat-endpoint'

print(f"Testing endpoint: {endpoint_name}")

## API Reference

The endpoint expects a JSON payload with:

| Field | Type | Description |
|-------|------|-------------|
| `temperature` | float | Current machine temperature (°C) |
| `room_temp` | float | Ambient room temperature (°C) |

**Response**:

| Field | Type | Description |
|-------|------|-------------|
| `prediction` | int | 0 = Normal, 1 = Overheat |
| `probability` | float | Probability of overheating (0.0 - 1.0) |

**Note**: The model uses a threshold of 80°C for classifying overheat.

## Helper Function

Let's create a helper function to call the endpoint:

In [ ]:
def predict_overheat(temperature: float, room_temp: float) -> dict:
    """
    Call the machine overheat prediction endpoint.
    
    Args:
        temperature: Machine temperature in Celsius
        room_temp: Room temperature in Celsius
    
    Returns:
        dict with 'prediction' (0/1) and 'probability' (0.0-1.0)
    """
    payload = json.dumps({
        'temperature': temperature,
        'room_temp': room_temp
    })
    
    response = runtime.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType='application/json',
        Body=payload
    )
    
    result = json.loads(response['Body'].read().decode())
    return result

# Quick test
result = predict_overheat(75, 25)
print(f"Test call successful: {result}")

## Test Scenario 1: Normal Operating Temperatures

Test machines running within safe parameters (< 80°C):

In [ ]:
normal_temps = [
    (65, 22, "Cool machine, normal room"),
    (70, 25, "Warm machine, normal room"),
    (75, 24, "Moderately warm machine"),
    (78, 23, "Near threshold but safe"),
]

print("=== Normal Operating Temperatures ===")
print(f"{'Temp':>6} | {'Room':>6} | {'Pred':>6} | {'Prob':>8} | Description")
print("-" * 60)

for temp, room, desc in normal_temps:
    result = predict_overheat(temp, room)
    status = "⚠️ OVERHEAT" if result['prediction'] == 1 else "✓ Normal"
    print(f"{temp:>5}°C | {room:>5}°C | {status:>10} | {result['probability']:>7.1%} | {desc}")

## Test Scenario 2: Overheating Machines

Test machines exceeding the 80°C threshold:

In [ ]:
overheat_temps = [
    (81, 25, "Just above threshold"),
    (85, 24, "Moderate overheat"),
    (90, 26, "Significant overheat"),
    (95, 25, "Critical temperature"),
]

print("=== Overheating Machines ===")
print(f"{'Temp':>6} | {'Room':>6} | {'Pred':>6} | {'Prob':>8} | Description")
print("-" * 60)

for temp, room, desc in overheat_temps:
    result = predict_overheat(temp, room)
    status = "⚠️ OVERHEAT" if result['prediction'] == 1 else "✓ Normal"
    print(f"{temp:>5}°C | {room:>5}°C | {status:>10} | {result['probability']:>7.1%} | {desc}")

## Test Scenario 3: Borderline Cases

Test temperatures near the 80°C decision boundary:

In [ ]:
borderline_temps = [
    (79, 25, "Just below threshold"),
    (79.5, 25, "Very close to threshold"),
    (80, 25, "Exactly at threshold"),
    (80.5, 25, "Just above threshold"),
]

print("=== Borderline Cases (near 80°C) ===")
print(f"{'Temp':>6} | {'Room':>6} | {'Pred':>6} | {'Prob':>8} | Description")
print("-" * 60)

for temp, room, desc in borderline_temps:
    result = predict_overheat(temp, room)
    status = "⚠️ OVERHEAT" if result['prediction'] == 1 else "✓ Normal"
    print(f"{temp:>5}°C | {room:>5}°C | {status:>10} | {result['probability']:>7.1%} | {desc}")

## Test Scenario 4: Room Temperature Impact

Test how different room temperatures affect the prediction for the same machine temperature:

In [ ]:
machine_temp = 78  # Fixed machine temperature
room_temps = [18, 20, 22, 24, 26, 28, 30]

print(f"=== Room Temperature Impact (Machine at {machine_temp}°C) ===")
print(f"{'Room':>6} | {'TempDiff':>8} | {'Pred':>6} | {'Prob':>8}")
print("-" * 45)

for room in room_temps:
    result = predict_overheat(machine_temp, room)
    temp_diff = machine_temp - room
    status = "⚠️" if result['prediction'] == 1 else "✓"
    print(f"{room:>5}°C | {temp_diff:>7}°C | {status:>6} | {result['probability']:>7.1%}")

print("\nNote: Higher temp_diff increases overheat probability")

## Test Scenario 5: Batch Predictions

Test multiple machines at once and create a summary report:

In [ ]:
# Simulate 5 machines with current readings
machines = [
    {'machine_id': 'M1', 'temperature': 72, 'room_temp': 24},
    {'machine_id': 'M2', 'temperature': 85, 'room_temp': 25},
    {'machine_id': 'M3', 'temperature': 68, 'room_temp': 23},
    {'machine_id': 'M4', 'temperature': 91, 'room_temp': 26},
    {'machine_id': 'M5', 'temperature': 77, 'room_temp': 24},
]

# Get predictions for all machines
results = []
for machine in machines:
    pred = predict_overheat(machine['temperature'], machine['room_temp'])
    results.append({
        'Machine': machine['machine_id'],
        'Temperature': f"{machine['temperature']}°C",
        'Room Temp': f"{machine['room_temp']}°C",
        'Status': '⚠️ OVERHEAT' if pred['prediction'] == 1 else '✓ Normal',
        'Probability': f"{pred['probability']*100:.1f}%"
    })

# Display as table
df_results = pd.DataFrame(results)
print("=== Machine Status Report ===")
display(df_results)

# Summary
overheating = sum(1 for r in results if 'OVERHEAT' in r['Status'])
print(f"\nSummary: {overheating} of {len(machines)} machines are overheating!")

## Test Scenario 6: Raw API Call

Demonstrate the raw boto3 API call for integration with other systems:

In [ ]:
# This is how external applications would call the endpoint
import json

# Prepare the payload
payload = json.dumps({
    'temperature': 82.5,
    'room_temp': 24.0
})

print("Request:")
print(f"  Endpoint: {endpoint_name}")
print(f"  Content-Type: application/json")
print(f"  Body: {payload}")

# Make the API call
response = runtime.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType='application/json',
    Body=payload
)

# Parse the response
result = json.loads(response['Body'].read().decode())

print("\nResponse:")
print(f"  HTTP Status: {response['ResponseMetadata']['HTTPStatusCode']}")
print(f"  Content-Type: {response['ContentType']}")
print(f"  Body: {json.dumps(result)}")

## Key Takeaways

**Model Behavior**:
- The model predicts overheat when temperature exceeds ~80°C
- The `temp_diff` (temperature - room_temp) is a key feature
- Probabilities near 50% indicate borderline cases

**API Integration**:
```python
# Minimal code to call the endpoint
import boto3, json
runtime = boto3.client('sagemaker-runtime', region_name='eu-west-1')

response = runtime.invoke_endpoint(
    EndpointName='machine-overheat-endpoint',
    ContentType='application/json',
    Body=json.dumps({'temperature': 85, 'room_temp': 25})
)
result = json.loads(response['Body'].read().decode())
# result = {'prediction': 1, 'probability': 0.95}
```

**Production Considerations**:
- Add error handling for network/service issues
- Implement retries with exponential backoff
- Monitor endpoint latency and errors via CloudWatch
- Consider batch transform for high-volume predictions

## Cleanup Reminder

**Important**: Remember to delete the endpoint when you're done to avoid charges!

```python
# Run in notebook 08 or via AWS CLI:
# aws sagemaker delete-endpoint --endpoint-name machine-overheat-endpoint --region eu-west-1
```